# NFL ATS ROI

Purpose: reproduce the portfolio `/api/sports-edges/ats-weekly` calculation for NFL and turn it into notebook-backed evidence.

This notebook joins scored Supabase `games` rows to the latest `model_predictions`, computes ATS wins/losses/pushes using the same home-margin convention as the portfolio API, and summarizes flat-unit ROI at -110.

Headline ATS uses `my_spread` vs final margin for every graded game. **Edge bucket tables** use only rows with a numeric `book_spread` (otherwise `edge_vs_book` is undefined). The summary dict includes `graded_missing_book_spread` and `ats_where_book_line_present` for the filtered subset.


In [1]:
LEAGUE = "NFL"
SEASON = None
EDGE_BUCKETS = [-99, -8, -5, -3, -1, 1, 3, 5, 8, 99]
WIN_PROFIT_AT_MINUS_110 = 100 / 110


In [2]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
DATA_CORE = ROOT.parent if ROOT.name == "notebooks" else ROOT
if str(DATA_CORE) not in sys.path:
    sys.path.insert(0, str(DATA_CORE))

from src.utils.supabase_pg import create_pg_connection, load_supabase_credentials

def infer_current_season(league: str) -> int:
    now = datetime.now(timezone.utc)
    return now.year - 1 if now.month < 8 else now.year

season = SEASON or infer_current_season(LEAGUE)
season


2025

In [3]:
QUERY = """
WITH latest_predictions AS (
  SELECT p.game_id, p.my_spread, p.my_home_win_prob, p.model_version, p.asof_ts,
         ROW_NUMBER() OVER (PARTITION BY p.game_id ORDER BY p.asof_ts DESC) AS rn
  FROM model_predictions p
)
SELECT g.id AS game_id, g.league, g.season, g.week, g.game_time_utc,
       g.home_team, g.away_team, g.book_spread, g.home_score, g.away_score,
       p.my_spread, p.my_home_win_prob, p.model_version, p.asof_ts
FROM games g
JOIN latest_predictions p ON p.game_id = g.id AND p.rn = 1
WHERE g.league = %s
  AND g.season = %s
  AND g.home_score IS NOT NULL
  AND g.away_score IS NOT NULL
ORDER BY g.game_time_utc
"""

def load_scored_predictions(league: str, season: int) -> pd.DataFrame:
    creds = load_supabase_credentials()
    missing = [key for key in ("url", "db_password") if not creds.get(key)]
    if missing:
        print(f"Missing Supabase credentials: {missing}. Returning empty frame.")
        return pd.DataFrame()
    conn = create_pg_connection(
        supabase_url=creds["url"],
        password=creds["db_password"],
        host_override=creds.get("db_host"),
        port=creds["db_port"],
        database=creds["db_name"],
        user=creds["db_user"],
    )
    try:
        return pd.read_sql_query(QUERY, conn, params=(league, season))
    finally:
        conn.close()

raw = load_scored_predictions(LEAGUE, season)
raw.head()


/tmp/ipykernel_65206/790095388.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(QUERY, conn, params=(league, season))


,game_id,league,season,week,game_time_utc,home_team,away_team,book_spread,home_score,away_score,my_spread,my_home_win_prob,model_version,asof_ts
0,cc963476-f1cb-4aeb-be88-a83b5780e370,NFL,2025,8,2025-10-23 00:00:00+00:00,LAC,MIN,-3.0,37,10,-2.680210,0.569087,v2,2025-11-18 01:07:59.891509+00:00
1,db47ebb8-8b41-4b93-a232-fffcd5b5760e,NFL,2025,8,2025-10-26 00:00:00+00:00,IND,TEN,-14.5,38,14,-18.835747,0.892288,v2,2025-11-18 01:07:59.891509+00:00
2,b55e8123-2ee1-4e18-824c-ccb8919b71c6,NFL,2025,8,2025-10-26 00:00:00+00:00,CAR,BUF,7.5,9,40,-0.171786,0.441615,v2,2025-11-18 01:07:59.891509+00:00
3,0c2dfbef-815f-4154-bcc1-ff68517a6612,NFL,2025,8,2025-10-26 00:00:00+00:00,HOU,SF,-3.0,26,15,-2.191209,0.507286,v2,2025-11-18 01:07:59.891509+00:00
4,12051103-b97d-42d9-a42f-67ff1e587740,NFL,2025,8,2025-10-26 00:00:00+00:00,ATL,MIA,-7.0,10,34,-7.511415,0.617138,v2,2025-11-18 01:07:59.891509+00:00


In [4]:
def add_ats_columns(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    out = df.copy()
    for col in ["home_score", "away_score", "my_spread", "book_spread", "my_home_win_prob"]:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    out["actual_margin"] = out["home_score"] - out["away_score"]
    out["cover_margin"] = out["actual_margin"] + out["my_spread"]
    out["ats_result"] = np.select(
        [out["cover_margin"].abs() < 1e-3, out["cover_margin"] > 0],
        ["push", "win"],
        default="loss",
    )
    out["edge_vs_book"] = out["my_spread"] - out["book_spread"]
    out["home_win_actual"] = (out["actual_margin"] > 0).astype(int)
    return out

games = add_ats_columns(raw)
games[["game_time_utc", "away_team", "home_team", "book_spread", "my_spread", "actual_margin", "ats_result"]].head() if not games.empty else games


,game_time_utc,away_team,home_team,book_spread,my_spread,actual_margin,ats_result
0,2025-10-23 00:00:00+00:00,MIN,LAC,-3.0,-2.680210,27,win
1,2025-10-26 00:00:00+00:00,TEN,IND,-14.5,-18.835747,24,win
2,2025-10-26 00:00:00+00:00,BUF,CAR,7.5,-0.171786,-31,loss
3,2025-10-26 00:00:00+00:00,SF,HOU,-3.0,-2.191209,11,win
4,2025-10-26 00:00:00+00:00,MIA,ATL,-7.0,-7.511415,-24,loss


In [5]:
def summarize_ats(df: pd.DataFrame) -> dict:
    if df.empty:
        return {"league": LEAGUE, "season": season, "graded_games": 0, "note": "No scored games with predictions available."}
    wins = int((df["ats_result"] == "win").sum())
    losses = int((df["ats_result"] == "loss").sum())
    pushes = int((df["ats_result"] == "push").sum())
    risked = wins + losses
    net_units = wins * WIN_PROFIT_AT_MINUS_110 - losses
    return {
        "league": LEAGUE,
        "season": season,
        "wins": wins,
        "losses": losses,
        "pushes": pushes,
        "graded_games": wins + losses + pushes,
        "ats_pct": wins / risked if risked else None,
        "flat_roi_at_minus_110": net_units / risked if risked else None,
        "net_units": net_units,
    }


summary = summarize_ats(games)
if not games.empty:
    summary["graded_missing_book_spread"] = int(games["book_spread"].isna().sum())
    book_present = games[games["book_spread"].notna()]
    summary["ats_where_book_line_present"] = summarize_ats(book_present) if not book_present.empty else None

summary


{'league': 'NFL',
 'season': 2025,
 'wins': 27,
 'losses': 29,
 'pushes': 0,
 'graded_games': 56,
 'ats_pct': 0.48214285714285715,
 'flat_roi_at_minus_110': -0.07954545454545459,
 'net_units': -4.454545454545457,
 'graded_missing_book_spread': 0,
 'ats_where_book_line_present': {'league': 'NFL',
  'season': 2025,
  'wins': 27,
  'losses': 29,
  'pushes': 0,
  'graded_games': 56,
  'ats_pct': 0.48214285714285715,
  'flat_roi_at_minus_110': -0.07954545454545459,
  'net_units': -4.454545454545457}}

In [6]:
if not games.empty:
    games_for_edge = games[games["book_spread"].notna()].copy()
    games_for_edge["edge_bucket"] = pd.cut(games_for_edge["edge_vs_book"], bins=EDGE_BUCKETS)
    edge_summary = (
        games_for_edge.groupby("edge_bucket", observed=True)
        .agg(
            games=("game_id", "count"),
            wins=("ats_result", lambda s: int((s == "win").sum())),
            losses=("ats_result", lambda s: int((s == "loss").sum())),
            pushes=("ats_result", lambda s: int((s == "push").sum())),
            avg_edge=("edge_vs_book", "mean"),
            avg_cover_margin=("cover_margin", "mean"),
        )
        .reset_index()
    )
    edge_summary["risked_games"] = edge_summary["wins"] + edge_summary["losses"]
    edge_summary["ats_pct"] = edge_summary["wins"] / edge_summary["risked_games"].replace(0, np.nan)
    edge_summary["roi"] = (edge_summary["wins"] * WIN_PROFIT_AT_MINUS_110 - edge_summary["losses"]) / edge_summary["risked_games"].replace(0, np.nan)
else:
    edge_summary = pd.DataFrame()

edge_summary


,edge_bucket,games,wins,losses,pushes,avg_edge,avg_cover_margin,risked_games,ats_pct,roi
0,"(-99, -8]",5,2,3,0,-19.840544,-2.340544,5,0.400000,-0.236364
1,"(-8, -5]",4,2,2,0,-6.583558,-7.208558,4,0.500000,-0.045455
2,"(-5, -3]",9,3,6,0,-4.216413,-6.549746,9,0.333333,-0.363636
3,"(-3, -1]",7,2,5,0,-1.787178,-1.358607,7,0.285714,-0.454545
4,"(-1, 1]",13,7,6,0,-0.207766,2.061465,13,0.538462,0.027972
5,"(1, 3]",9,4,5,0,1.877084,0.321529,9,0.444444,-0.151515
6,"(3, 5]",7,5,2,0,3.817069,5.745640,7,0.714286,0.363636
7,"(5, 8]",2,2,0,0,5.934030,3.684030,2,1.000000,0.909091


In [7]:
if not games.empty:
    display_cols = ["game_time_utc", "away_team", "home_team", "book_spread", "my_spread", "actual_margin", "cover_margin", "model_version"]
    print("Biggest hits")
    display(games.sort_values("cover_margin", ascending=False).head(10)[display_cols])
    print("Biggest misses")
    display(games.sort_values("cover_margin", ascending=True).head(10)[display_cols])


Biggest hits


,game_time_utc,away_team,home_team,book_spread,my_spread,actual_margin,cover_margin,model_version
54,2025-11-16 00:00:00+00:00,LAC,JAX,3.0,1.082170,29,30.082170,v2
31,2025-11-09 00:00:00+00:00,BUF,MIA,8.0,8.224751,17,25.224751,v2
0,2025-10-23 00:00:00+00:00,MIN,LAC,-3.0,-2.680210,27,24.319790,v2
43,2025-11-16 00:00:00+00:00,CIN,PIT,-5.5,-4.132762,22,17.867238,v2
33,2025-11-09 00:00:00+00:00,ARI,SEA,-7.5,-5.133820,22,16.866180,v2
34,2025-11-09 00:00:00+00:00,PIT,LAC,-3.0,1.142578,15,16.142578,v2
9,2025-10-26 00:00:00+00:00,DAL,DEN,-4.0,-4.222938,20,15.777062,v2
12,2025-10-27 00:00:00+00:00,WAS,KC,-10.0,-6.716917,21,14.283083,v2
22,2025-11-02 00:00:00+00:00,IND,PIT,3.0,6.032491,7,13.032491,v2
10,2025-10-26 00:00:00+00:00,CHI,BAL,-2.5,-3.067252,14,10.932748,v2


Biggest misses


,game_time_utc,away_team,home_team,book_spread,my_spread,actual_margin,cover_margin,model_version
4,2025-10-26 00:00:00+00:00,MIA,ATL,-7.0,-7.511415,-24,-31.511415,v2
2,2025-10-26 00:00:00+00:00,BUF,CAR,7.5,-0.171786,-31,-31.171786,v2
13,2025-10-30 00:00:00+00:00,BAL,MIA,7.5,2.562334,-22,-19.437666,v2
14,2025-11-02 00:00:00+00:00,SEA,WAS,3.0,4.750474,-24,-19.249526,v2
48,2025-11-16 00:00:00+00:00,SF,ARI,3.0,0.310896,-19,-18.689104,v2
29,2025-11-09 00:00:00+00:00,DET,WAS,8.5,6.893985,-22,-15.106015,v2
16,2025-11-02 00:00:00+00:00,MIN,DET,9.5,-12.003050,-3,-15.003050,v2
37,2025-11-09 00:00:00+00:00,NO,CAR,-5.5,-4.570561,-10,-14.570561,v2
28,2025-11-09 00:00:00+00:00,LA,SF,6.0,2.478695,-16,-13.521305,v2
26,2025-11-03 00:00:00+00:00,ARI,DAL,3.0,-1.951849,-10,-11.951849,v2


In [8]:
if not games.empty and games["my_home_win_prob"].notna().sum() >= 20:
    games["win_prob_bucket"] = pd.cut(games["my_home_win_prob"], bins=np.linspace(0, 1, 11), include_lowest=True)
    calibration = (
        games.groupby("win_prob_bucket", observed=True)
        .agg(games=("game_id", "count"), avg_pred=("my_home_win_prob", "mean"), actual_home_win_rate=("home_win_actual", "mean"))
        .reset_index()
    )
else:
    calibration = pd.DataFrame({"note": ["Not enough completed games with win probabilities for calibration buckets."]})

calibration


,win_prob_bucket,games,avg_pred,actual_home_win_rate
0,"(0.2, 0.3]",4,0.260843,0.250000
1,"(0.3, 0.4]",8,0.349196,0.125000
2,"(0.4, 0.5]",10,0.457516,0.500000
3,"(0.5, 0.6]",16,0.531068,0.437500
4,"(0.6, 0.7]",8,0.629653,0.875000
5,"(0.7, 0.8]",7,0.747853,0.571429
6,"(0.8, 0.9]",3,0.842828,1.000000
